# RR SSC-MEK PHASE B1: rosters, signatures, compound IDs, freeze (NO SCORING)
Pre-registration committed 2026-10-01 22:24 EDT, SHA-256 `45de35e26156bf1c70d22bc9100adaeec30f3fc4e70b803e075c124960c3a4c9`.

This notebook computes **no connectivity score** and never opens the library matrix. It builds the rosters by field name, builds the disease signatures, checks the compound identifiers, and freezes everything with a hash. The freeze report is read before the scoring notebook is written.

P01's source data (GSE138669, GSE195452) are never fetched.

Requires Phase A to have run in the same folder: it leaves the LINCS metadata and the series matrix files there.

Run: hamburger (top left) > Runtime > Run all. Then copy the text between `=== SSC-MEK PHASE B1 REPORT BEGIN ===` and `=== SSC-MEK PHASE B1 REPORT END ===` and paste it back. **If a cell stops with an error, paste the error instead.** A stop is a guard firing, not a crash to work around.

In [ ]:
# CELL 1: setup. Reads the benchmark folder for the library gene order; writes only to ssc_mek.
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, sys
BENCH = "/content/drive/MyDrive/RitschelResearch/benchmark"
W = "/content/drive/MyDrive/RitschelResearch/ssc_mek"; os.makedirs(W, exist_ok=True); os.chdir(W)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit"], check=False)
import numpy as np, pandas as pd, gzip, re, urllib.request, io, hashlib, json, collections, time
PREREG = "45de35e26156bf1c70d22bc9100adaeec30f3fc4e70b803e075c124960c3a4c9"
REPORT = []
def R(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); REPORT.append(s)
for f in ["geneinfo_beta.txt", "compoundinfo_beta.txt", "siginfo_beta.txt"]:
    assert os.path.exists(f), "RUN PHASE A FIRST: missing " + f
g = pd.read_csv("geneinfo_beta.txt", sep="\t")
LMG = g.loc[g.feature_space == "landmark"].copy(); LMG["gene_id"] = LMG.gene_id.astype(int)
genes = np.load(os.path.join(BENCH, "libA_genes.npy")); GI = {int(x): i for i, x in enumerate(genes)}
assert len(genes) == 978 and set(GI) == set(LMG.gene_id), "LIBRARY GENE ORDER IS NOT THE 978 LANDMARKS"
SYM2E = {str(s).upper(): int(e) for e, s in zip(LMG.gene_id, LMG.gene_symbol)}
ENS2E = {}
if "ensembl_id" in LMG.columns:
    for e, x in zip(LMG.gene_id, LMG.ensembl_id.astype(str)):
        for y in x.split("|"):
            if y.startswith("ENSG"): ENS2E[y.split(".")[0]] = int(e)
R("setup: 978 landmarks in library order; symbol map", len(SYM2E), "ensembl map", len(ENS2E))
R("prereg", PREREG)

In [ ]:
# CELL 2: compound identifiers. Every hardcoded ID is checked against the library's own name for it and
# against the signature count Phase A reported. PD-0325901's formula must match or the run stops.
from rdkit import Chem
from rdkit.Chem.rdMolDescriptors import CalcMolFormula
ci = pd.read_csv("compoundinfo_beta.txt", sep="\t", dtype=str).fillna("")
si = pd.read_csv("siginfo_beta.txt", sep="\t",
                 usecols=["sig_id", "pert_id", "pert_type", "is_exemplar_sig", "qc_pass", "cell_iname"], low_memory=False)
si = si[(si.pert_type == "trt_cp") & (si.is_exemplar_sig == 1) & (si.qc_pass == 1)]
NS = si.groupby("pert_id").size().to_dict()
ASCN = si[si.cell_iname == "ASC"].groupby("pert_id").size().to_dict()
def formula(pid):
    sm = [x for x in ci.loc[ci.pert_id == pid, "canonical_smiles"] if x]
    if not sm: return "NO_SMILES"
    m = Chem.MolFromSmiles(sm[0])
    if m is None: return "BAD_SMILES"
    return CalcMolFormula(max(Chem.GetMolFrags(m, asMols=True), key=lambda x: x.GetNumHeavyAtoms()))
IDS = [("primary", "PD-0325901", "BRD-K49865102", 25), ("primary", "PD-0325901", "BRD-A50454580", 8),
       ("class", "selumetinib", "BRD-K57080016", 34), ("class", "trametinib", "BRD-K12343256", 22),
       ("class", "PD-184352", "BRD-K05104363", 17), ("class", "U-0126", "BRD-K18787491", 17),
       ("class", "U-0126", "BRD-K46419649", 12), ("class", "U-0126", "BRD-K91701654", 4),
       ("class", "PD-98059", "BRD-K62810658", 15), ("class", "AS-703026", "BRD-K89014967", 14),
       ("class", "refametinib", "BRD-K85751432", 12), ("class", "Ro-4987655", "BRD-K64538373", 11),
       ("class", "TAK-733", "BRD-K26667523", 9), ("class", "PD-198306", "BRD-K88677950", 9),
       ("reference", "dasatinib", "BRD-K49328571", 29), ("reference", "bosutinib", "BRD-K99964838", 25),
       ("reference", "nintedanib", "BRD-K49075727", 13), ("reference", "mycophenolate-mofetil", "BRD-K92428153", 20)]
PD_FORMULA = "C16H14F3IN2O4"
DR = []
for role, drug, pid, n_exp in IDS:
    names = set(ci.loc[ci.pert_id == pid, "cmap_name"])
    assert names == {drug}, ("ID DOES NOT CARRY THE EXPECTED NAME", pid, drug, names)
    n = int(NS.get(pid, 0))
    assert n == n_exp, ("SIGNATURE COUNT DIFFERS FROM PHASE A", pid, n, n_exp)
    fm = formula(pid)
    if drug == "PD-0325901":
        assert fm == PD_FORMULA, ("PD-0325901 FORMULA MISMATCH", pid, fm, PD_FORMULA)
    DR.append(dict(role=role, drug=drug, pert_id=pid, formula=fm, n_lib_sigs=n, n_asc_sigs=int(ASCN.get(pid, 0))))
    R("compound %-9s %-22s %s %-16s sigs %3d ASC %d" % (role, drug, pid, fm, n, int(ASCN.get(pid, 0))))
CLS = [d for d in DR if d["role"] in ("primary", "class")]
assert len(CLS) == 14 and len(set(d["drug"] for d in CLS)) == 11, "MEK1/2 CLASS IS NOT 11 DRUGS IN 14 IDS"
for drug in sorted(set(d["drug"] for d in CLS)):
    fs = sorted(set(d["formula"] for d in CLS if d["drug"] == drug))
    if len(fs) > 1: R("NOTE: identifiers of", drug, "do not share one formula:", fs)
pd.DataFrame(DR).to_csv("sscmek_drugs_frozen.csv", index=False)
assert len(pd.read_csv("sscmek_drugs_frozen.csv")) == len(DR), "DRUG TABLE READ-BACK MISMATCH"
R("compounds frozen:", len(DR), "IDs; MEK1/2 class 11 drugs in 14 IDs; PD-0325901 formula", PD_FORMULA, "confirmed")

In [ ]:
# CELL 3: shared helpers. stat() is the benchmark's, unchanged.
UA = {"User-Agent": "Mozilla/5.0 (research; contact glen@ritschelresearch.com)"}
def fetch_to(u, path, timeout=900):
    if not os.path.exists(path):
        b = urllib.request.urlopen(urllib.request.Request(u, headers=UA), timeout=timeout).read()
        open(path, "wb").write(b)
    return path
def strip(x): return x.strip().strip('"').strip()
def series_meta(stem):
    f = stem + "_series_matrix.txt.gz"
    assert os.path.exists(f), "MISSING " + f + " (run Phase A first)"
    out = []
    with gzip.open(f, "rt", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if line.startswith("!series_matrix_table_begin"): break
            out.append(line)
    return out
def fields_of(meta):
    acc = [l for l in meta if l.startswith("!Sample_geo_accession")]
    assert acc, "NO SAMPLE ACCESSION LINE"
    gsm = [strip(x) for x in acc[0].rstrip("\n").split("\t")[1:]]
    F_ = collections.OrderedDict()
    for l in meta:
        if l.startswith("!Sample_characteristics_ch1"):
            for i, v in enumerate(strip(x) for x in l.rstrip("\n").split("\t")[1:]):
                if ":" in v:
                    k, _, val = v.partition(":")
                    F_.setdefault(k.strip().lower(), {})[i] = val.strip()
    return gsm, F_
def stat(E, a, b, k=150):
    A = E[:, a]; B = E[:, b]; d = A.mean(1) - B.mean(1)
    pooled = np.sqrt(0.5 * (A.var(1, ddof=1) + B.var(1, ddof=1)))
    fl = np.median(pooled[pooled > 0]) if np.any(pooled > 0) else 1.0
    st = d / np.maximum(pooled, fl * 0.1)
    o = np.argsort(-st, kind="stable")
    return o[:k].copy(), o[-k:].copy()
SIG = {}
def build(qid, series, E, grp, note):
    keep = ~np.isnan(E).any(axis=1); idx = np.where(keep)[0]
    cases = [i for i, x in enumerate(grp) if x == "case"]
    ctrls = [i for i, x in enumerate(grp) if x == "control"]
    assert len(cases) > 0 and len(ctrls) > 0, "EMPTY ARM IS FATAL"
    assert int(keep.sum()) >= 300, ("TOO FEW USABLE LANDMARKS", int(keep.sum()))
    up, dn = stat(E[keep], cases, ctrls)
    SIG[qid] = dict(series=series, n_case=len(cases), n_ctrl=len(ctrls), genes_usable=int(keep.sum()),
                    up=[int(genes[idx[i]]) for i in up], dn=[int(genes[idx[i]]) for i in dn],
                    usable=[int(genes[i]) for i in idx], note=note)
    R("%s %s: cases %d controls %d usable landmarks %d | %s" % (qid, series, len(cases), len(ctrls), int(keep.sum()), note))
R("helpers ready: statistic is the benchmark's pooled-sd difference with a 0.1x median floor, k=150")

In [ ]:
# CELL 4: platform annotation. A cached or downloaded file is NEVER trusted by name: it is accepted only
# if it actually contains a platform table. Sources are tried in order and every rejection is reported
# with the first characters of what came back, so a bad file is diagnosed rather than guessed at.
# (The first B1 run failed here: a file left by Phase A was accepted on its name alone.)
def _has_table(txt): return "!platform_table_begin" in txt
def platform_table(plat):
    cache = plat + "_table.txt"
    if os.path.exists(cache):
        t = open(cache, encoding="utf-8", errors="replace").read()
        if _has_table(t): R("platform", plat, "table from cache", cache, "(%d chars)" % len(t)); return t, "cache"
        os.remove(cache); R("platform", plat, "cached table rejected: no table marker; refetching")
    tries = []
    gz = plat + ".annot.gz"
    if os.path.exists(gz):
        try:
            t = gzip.open(gz, "rt", encoding="utf-8", errors="replace").read()
            if _has_table(t): open(cache, "w", encoding="utf-8").write(t); R("platform", plat, "from a local", gz); return t, "local annot.gz"
            tries.append("local %s: no table marker, %d chars, starts %r" % (gz, len(t), t[:120]))
        except Exception as e: tries.append("local %s: %s" % (gz, type(e).__name__))
        os.remove(gz); R("platform", plat, "removed the unusable local", gz)
    SRC = [("annot.gz", "https://ftp.ncbi.nlm.nih.gov/geo/platforms/%snnn/%s/annot/%s.annot.gz" % (plat[:-3], plat, plat)),
           ("acc.cgi full", "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=%s&targ=self&form=text&view=full" % plat),
           ("acc.cgi data", "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=%s&targ=self&form=text&view=data" % plat)]
    for label, u in SRC:
        try:
            raw = urllib.request.urlopen(urllib.request.Request(u, headers=UA), timeout=900).read()
            t = gzip.decompress(raw).decode("utf-8", "replace") if raw[:2] == b"\x1f\x8b" else raw.decode("utf-8", "replace")
            if _has_table(t):
                open(cache, "w", encoding="utf-8").write(t)
                R("platform", plat, "table from", label, "(%d chars)" % len(t)); return t, label
            tries.append("%s: no table marker, %d chars, starts %r" % (label, len(t), t[:120]))
        except Exception as e:
            tries.append("%s: %s %s" % (label, type(e).__name__, str(e)[:80]))
        R("platform", plat, "rejected", label, "->", tries[-1])
    raise AssertionError(("NO PLATFORM TABLE FOR " + plat, tries))
def probe_to_entrez(plat):
    txt, src = platform_table(plat)
    lines = txt.splitlines(); i = 0
    while i < len(lines) and not lines[i].startswith("!platform_table_begin"): i += 1
    hdr = [c.strip() for c in lines[i + 1].split("\t")]
    cand_e = [j for j, c in enumerate(hdr) if c.upper() in ("GENE ID", "GENE", "ENTREZ_GENE_ID", "ENTREZ GENE ID", "GENE_ID", "LOCUSLINK_ID")]
    cand_s = [j for j, c in enumerate(hdr) if c.upper() in ("GENE SYMBOL", "GENE_SYMBOL", "SYMBOL", "ORF")]
    R("platform", plat, "columns:", hdr[:14], "| entrez col", [hdr[j] for j in cand_e], "| symbol col", [hdr[j] for j in cand_s])
    assert cand_e or cand_s, ("NO GENE COLUMN ON " + plat, hdr)
    m = {}; via = collections.Counter()
    for ln in lines[i + 2:]:
        if ln.startswith("!platform_table_end"): break
        q = ln.rstrip("\n").split("\t")
        if not q or not q[0]: continue
        e = None
        for j in cand_e:
            if len(q) > j:
                x = q[j].split("///")[0].strip()
                if x.isdigit(): e = int(x); via["entrez"] += 1; break
        if e is None:
            for j in cand_s:
                if len(q) > j:
                    e = SYM2E.get(q[j].split("///")[0].strip().upper())
                    if e: via["symbol"] += 1; break
        if e in GI: m[q[0].strip()] = e
    R("platform", plat, "source", src, "| probes mapped to a landmark:", len(m), "| distinct landmarks:", len(set(m.values())), "of 978 | by", dict(via))
    PLATSRC[plat] = src
    return m
PLATSRC = {}
def matrix_values(stem, gsm):
    rows = []; hdr = None; inb = False
    with gzip.open(stem + "_series_matrix.txt.gz", "rt", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if line.startswith("!series_matrix_table_end"): break
            if inb:
                q = line.rstrip("\n").split("\t")
                if hdr is None: hdr = [strip(x) for x in q]
                else: rows.append(q)
                continue
            if line.startswith("!series_matrix_table_begin"): inb = True
    assert hdr is not None and hdr[1:] == gsm, "MATRIX COLUMNS DO NOT MATCH THE SAMPLE ACCESSIONS"
    return rows
def collapse(stem, gsm, plat):
    p2e = probe_to_entrez(plat)
    rows = matrix_values(stem, gsm)
    acc = {}
    for q in rows:
        e = p2e.get(strip(q[0]))
        if e is None: continue
        try: v = np.array([float(strip(x)) if strip(x) not in ("", "NA", "null", "NaN") else np.nan for x in q[1:]])
        except ValueError: continue
        acc.setdefault(GI[e], []).append(v)
    E = np.full((978, len(gsm)), np.nan)
    for gi_, L in acc.items(): E[gi_] = np.nanmedian(np.vstack(L), axis=0)
    fin = np.isfinite(E).all(axis=1)
    assert fin.any(), ("NO LANDMARK HAS VALUES IN EVERY SAMPLE FOR " + stem)
    p99 = np.nanpercentile(E[fin], 99)
    R(stem, "collapsed: rows in table", len(rows), "| landmarks with values", int(fin.sum()), "| 99th percentile %.3f" % p99)
    if p99 > 100:
        E = np.log2(np.maximum(E, 1)); R(stem, "99th percentile exceeds 100 -> log2(max(x,1)) applied [pre-registered scale rule]")
    else:
        R(stem, "99th percentile at or below 100 -> used as deposited [pre-registered scale rule]")
    return E

In [ ]:
# CELL 5: Q1 = GSE181549 (GENISOS registry). Groups by FIELD NAME, one biopsy per person.
stem = "GSE181549"; meta = series_meta(stem); gsm, FF = fields_of(meta)
R("Q1 fields:", list(FF))
plat = [strip(x) for l in meta if l.startswith("!Series_platform_id") for x in l.rstrip("\n").split("\t")[1:]]
assert plat == ["GPL13497"], ("Q1 PLATFORM CHANGED", plat)
cond = FF.get("condition", {}); num = FF.get("sample number within subject", {}); subj = FF.get("subject", {})
assert cond and num, "Q1 IS MISSING THE condition OR sample-number FIELD"
R("Q1 condition values:", dict(collections.Counter(cond.values())))
R("Q1 sample-number values:", dict(collections.Counter(num.values())))
sel = [i for i in range(len(gsm)) if num.get(i) == "1"]
grp_all = ["case" if cond.get(i) == "systemic sclerosis patient" else "control" if cond.get(i) == "Control" else "none" for i in range(len(gsm))]
unknown = [cond.get(i) for i in range(len(gsm)) if grp_all[i] == "none"]
assert not unknown, ("Q1 UNRECOGNIZED condition VALUES", collections.Counter(unknown))
grp = [grp_all[i] for i in sel]
cnt = collections.Counter(grp)
R("Q1 first biopsy per subject:", len(sel), "samples ->", dict(cnt))
assert cnt["case"] == 113 and cnt["control"] == 44, ("Q1 ARM COUNTS DIFFER FROM THE PRE-REGISTRATION 113/44", dict(cnt))
sj = [subj.get(i) for i in sel if grp_all[i] == "case"]
assert len(sj) == len(set(sj)), ("Q1 A SUBJECT APPEARS TWICE AMONG THE SELECTED CASES", len(sj), len(set(sj)))
R("Q1 distinct case subjects among the selected:", len(set(sj)))
E = collapse(stem, gsm, "GPL13497")
build("Q1", stem, E[:, sel], grp, "one biopsy per subject (sample number 1); groups from the condition field")

In [ ]:
# CELL 6: Q2 = GSE130955 (PRESS registry, RNA-seq). The series deposits no expression table, so this
# reads the per-sample supplementary files. If they hold only raw reads, Q2 is EXCLUDED on a deposit
# ground, exactly as the benchmark excluded its graft-versus-host row. Nothing is reprocessed.
stem = "GSE130955"; meta = series_meta(stem); gsm, FF = fields_of(meta)
R("Q2 fields:", list(FF))
supp = []
for l in meta:
    if l.startswith("!Sample_supplementary_file"):
        supp.append([strip(x) for x in l.rstrip("\n").split("\t")[1:]])
assert supp, "Q2 HAS NO SAMPLE SUPPLEMENTARY LINES"
assert all(len(row) == len(gsm) for row in supp), "Q2 SUPPLEMENTARY LINE WIDTH DOES NOT MATCH THE SAMPLES"
flat = [x if x and x.upper() != "NONE" else "" for x in supp[0]]
names = [x.split("/")[-1] for x in flat if x]
R("Q2 supplementary lines:", len(supp), "| files on the first line:", len([x for x in flat if x]), "of", len(gsm))
if len(supp) > 1: R("Q2 NOTE: samples carry", len(supp), "supplementary files each; only the first is read")
assert len(names) == len(gsm), "Q2 NOT EVERY SAMPLE HAS A SUPPLEMENTARY FILE"

R("Q2 first five:", names[:5])
exts = collections.Counter(re.sub(r"^GSM\d+_", "", n) for n in names)
R("Q2 filename shapes:", dict(list(exts.most_common(6))))
RAW = (".sra", ".fastq", ".fq", ".bam", ".cram")
Q2_RAW_ONLY = all(any(n.lower().endswith(e) or e[1:] in n.lower() for e in RAW) for n in names) if names else True
Q2_BUILT = False
if Q2_RAW_ONLY or not names:
    R("*** Q2 EXCLUDED ON A DEPOSIT GROUND: the deposit carries no gene-level expression, only raw reads.")
    R("*** Reprocessing would add aligner and normalization choices after the pre-registration. Not done.")
    R("*** The test proceeds on Q1 alone, as the pre-registration states.")
else:
    tabs = {}
    for i, x in enumerate(flat):
        n = x.split("/")[-1]
        assert n, ("Q2 SAMPLE HAS NO SUPPLEMENTARY FILE", gsm[i])
        u = "https://ftp.ncbi.nlm.nih.gov/geo/samples/%snnn/%s/suppl/%s" % (gsm[i][:-3], gsm[i], n)
        try: p = fetch_to(u, n, timeout=300)
        except Exception as e:
            R("Q2 FETCH FAILED for", gsm[i], n, type(e).__name__); tabs = {}; break
        op = gzip.open if n.endswith(".gz") else open
        d = {}
        with op(p, "rt", encoding="utf-8", errors="replace") as fh:
            for ln in fh:
                q = ln.rstrip("\n").split("\t")
                if len(q) < 2: continue
                try: d[q[0].strip()] = float(q[-1])
                except ValueError: continue
        assert len(d) > 5000, ("Q2 SUPPLEMENTARY FILE IS NOT A GENE TABLE", n, len(d))
        tabs[gsm[i]] = d
        if i % 20 == 0: R("  Q2 read", i + 1, "of", len(flat), "| rows in last file", len(d))
    assert tabs and len(tabs) == len(gsm), ("Q2 DID NOT READ ONE TABLE PER SAMPLE", len(tabs), len(gsm))
    ids = sorted(set().union(*[set(d) for d in tabs.values()]))
    R("Q2 union of gene identifiers:", len(ids), "| example", ids[:3])
    M = np.array([[tabs[s].get(k, 0.0) for s in gsm] for k in ids], dtype=float)
    lib = M.sum(0)                      # library size on the FULL matrix, before any gene subsetting
    R("Q2 library size per sample: min %d median %d max %d" % (int(lib.min()), int(np.median(lib)), int(lib.max())))
    assert lib.min() > 0, "Q2 A SAMPLE HAS ZERO LIBRARY SIZE"
    CPM = np.log2(M / np.maximum(lib, 1) * 1e6 + 1)
    acc = {}
    for i, k in enumerate(ids):
        e = ENS2E.get(k.split(".")[0]) or SYM2E.get(k.strip().upper())
        if e in GI: acc.setdefault(GI[e], []).append(CPM[i])
    E2 = np.full((978, len(gsm)), np.nan)
    for gi_, L in acc.items(): E2[gi_] = np.median(np.vstack(L), axis=0)
    R("Q2 landmarks with values:", int(np.isfinite(E2).all(axis=1).sum()), "of 978")
    dx = FF.get("diagnosis", {}); bn = FF.get("biopsy number for this subject", {})
    assert dx and bn, "Q2 IS MISSING THE diagnosis OR biopsy-number FIELD"
    R("Q2 diagnosis values:", dict(collections.Counter(dx.values())))
    sel2 = [i for i in range(len(gsm)) if bn.get(i) == "1"]
    g_all = ["case" if dx.get(i) == "SSc patient" else "control" if dx.get(i) == "healthy control" else "none" for i in range(len(gsm))]
    assert "none" not in [g_all[i] for i in sel2], "Q2 UNRECOGNIZED diagnosis VALUE AMONG SELECTED"
    grp2 = [g_all[i] for i in sel2]; c2 = collections.Counter(grp2)
    R("Q2 first biopsy per subject:", len(sel2), "samples ->", dict(c2))
    assert c2["case"] == 48 and c2["control"] == 33, ("Q2 ARM COUNTS DIFFER FROM THE PRE-REGISTRATION 48/33", dict(c2))
    build("Q2", stem, E2[:, sel2], grp2, "first biopsy per subject; log2 CPM, library size on the full matrix")
    Q2_BUILT = True

In [ ]:
# CELL 7: S1 = GSE95065, the declared sensitivity query. Reported, never gating.
stem = "GSE95065"; meta = series_meta(stem); gsm, FF = fields_of(meta)
plat = [strip(x) for l in meta if l.startswith("!Series_platform_id") for x in l.rstrip("\n").split("\t")[1:]]
R("S1 fields:", list(FF), "| platform", plat)
dx = FF.get("dx", {})
assert dx, "S1 IS MISSING THE dx FIELD"
R("S1 dx values:", dict(collections.Counter(dx.values())))
g1 = ["case" if dx.get(i) == "diffuse cutaneous systemic sclerosis" else "control" if dx.get(i) == "healthy" else "none" for i in range(len(gsm))]
assert "none" not in g1, ("S1 UNRECOGNIZED dx VALUE", collections.Counter(dx.values()))
c1_ = collections.Counter(g1)
assert c1_["case"] == 18 and c1_["control"] == 15, ("S1 ARM COUNTS DIFFER FROM PHASE A 18/15", dict(c1_))
R("S1 arms:", dict(c1_), "| batch field:", dict(collections.Counter(FF.get("batch", {}).values())))
S1_BUILT = False
try:
    E3 = collapse(stem, gsm, plat[0])
    build("S1", stem, E3, g1, "sensitivity only; same laboratory as the P01 source data, so individuals may overlap")
    S1_BUILT = True
except AssertionError as e:
    R("S1 NOT BUILT:", e)
except Exception as e:
    R("S1 NOT BUILT:", type(e).__name__, str(e)[:160])

In [ ]:
# CELL 8: freeze. Nothing here scores anything; the file is the input the scoring notebook will be locked to.
built = sorted(SIG)
assert "Q1" in SIG, "Q1 MUST BE BUILT"
out = {"prereg_sha256": PREREG, "k": 150,
       "statistic": "benchmark pooled-sd difference with a 0.1x median floor",
       "primary_queries": [q for q in built if q.startswith("Q")],
       "sensitivity_queries": [q for q in built if q.startswith("S")],
       "q2_built": bool(Q2_BUILT), "signatures": SIG,
       "drugs": pd.read_csv("sscmek_drugs_frozen.csv").to_dict(orient="records"),
       "excluded_by_design": ["GSE138669", "GSE195452"],
       "platform_annotation_source": PLATSRC}
b = json.dumps(out, sort_keys=True).encode()
open("sscmek_signatures_frozen_B1.json", "wb").write(b)
assert open("sscmek_signatures_frozen_B1.json", "rb").read() == b, "FREEZE READ-BACK MISMATCH"
h = hashlib.sha256(b).hexdigest()
print("=== SSC-MEK PHASE B1 REPORT BEGIN ===")
print("NO CONNECTIVITY SCORE WAS COMPUTED IN THIS NOTEBOOK")
print("\n".join(REPORT))
for q in built:
    v = SIG[q]
    print("SIG %s: %s cases %d controls %d usable %d up[:5] %s dn[:5] %s | %s" %
          (q, v["series"], v["n_case"], v["n_ctrl"], v["genes_usable"], v["up"][:5], v["dn"][:5], v["note"]))
print("PRIMARY QUERIES BUILT:", out["primary_queries"], "| Q2 built:", Q2_BUILT)
print("FROZEN sscmek_signatures_frozen_B1.json sha256", h)
print("=== SSC-MEK PHASE B1 REPORT END ===")